In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.4 MB/s eta 0:00:00


In [ ]:
import fitz
import re
import pandas as pd
from pathlib import Path
from datetime import datetime


class CVPipeline:
    # =========================
    # CONFIG
    # =========================
    FEATURE_HEADERS = {
        "title": ["title", "role", "position"],
        "summary": ["summary", "professional summary", "career summary", "profile", "professional profile"],
        "experience": [
            "experience", "work experience", "professional experience",
            "work history", "employment history"
        ],
        "skills": ["skills", "technical skills", "core skills", "highlights", "skill highlights"],
        "education": ["education", "education and training", "academic background", "training"]
    }

    MONTHS_PATTERN = r"(?:january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)"

    # =========================
    # CLEANING
    # =========================
    def clean_line(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        text = re.sub(r"[\u2022\u25cf\u25cb\u25aa\uf0b7\xb7\*\•]", " , ", text)
        text = text.encode("ascii", errors="ignore").decode()
        text = text.lower().replace("&", " and ")
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"( , \s*)+", ", ", text)

        return text.strip().strip(",")

    # =========================
    # HEADER LOGIC
    # =========================
    def is_header_line(self, line: str) -> bool:
        if not line or len(line) > 40:
            return False
        if line.endswith(".") or len(line.split()) > 5:
            return False
        return True

    def match_header(self, line: str):
        for feature, headers in self.FEATURE_HEADERS.items():
            for h in headers:
                if line == h or line.startswith(h + ":"):
                    return feature
        return None

    # =========================
    # TITLE FALLBACK
    # =========================
    def infer_title_from_experience(self, experience: str) -> str:
        if not experience:
            return ""
        first = re.sub(r"\b\d{4}\b", "", experience.split("|")[0])
        return " ".join(first.split()[:4])

    # =========================
    # PDF
    # =========================
    def pdf_to_text(self, pdf_path: Path) -> str:
        doc = fitz.open(pdf_path)
        return "\n".join(p.get_text("text") for p in doc)

    # =========================
    # FEATURE EXTRACTION
    # =========================
    def extract_features(self, text: str) -> dict:
        lines = text.split("\n")
        data = {k: "" for k in self.FEATURE_HEADERS}
        current = "title"
        title_taken = False

        for raw in lines:
            line = self.clean_line(raw)
            if not line:
                continue

            if self.is_header_line(line):
                matched = self.match_header(line)
                if matched:
                    current = matched
                    continue

            if current == "title" and not title_taken:
                data["title"] = line
                title_taken = True
                continue

            if current == "skills":
                sep = ", " if data[current] else ""
                data[current] += sep + line
            else:
                data[current] += " " + line

        return {k: v.strip() for k, v in data.items()}

    # =========================
    # EXPERIENCE ENRICH
    # =========================
    def calculate_duration(self, date_str):
        found = re.findall(
            rf"({self.MONTHS_PATTERN}\s+\d{{4}}|\d{{1,2}}/\d{{2,4}}|\b\d{{4}}\b|current|present|now)",
            date_str,
            re.IGNORECASE
        )

        def to_decimal(s):
            s = s.lower()
            if any(x in s for x in ["current", "present", "now"]):
                now = datetime.now()
                return now.year + now.month / 12

            if "/" in s:
                m, y = s.split("/")
                return int(y) + (int(m) / 12 if m.isdigit() else 0)

            m_y = re.search(rf"({self.MONTHS_PATTERN})\s+(\d{{4}})", s)
            if m_y:
                m_map = dict(zip(
                    ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"],
                    range(1,13)
                ))
                return int(m_y.group(2)) + m_map[m_y.group(1)[:3]] / 12

            return int(s) if s.isdigit() else None

        years = [to_decimal(x) for x in found if to_decimal(x)]
        return round(abs(years[-1] - years[0]), 1) if len(years) >= 2 else 0.5

    def enrich_experience(self, df):
        date_regex = rf"({self.MONTHS_PATTERN}\s+\d{{4}}|\d{{1,2}}/\d{{2,4}}|\b\d{{4}}\b)\s+(?:to|until|-)\s+({self.MONTHS_PATTERN}\s+\d{{4}}|\d{{1,2}}/\d{{2,4}}|\b\d{{4}}\b|current|present|now)"

        def process(row):
            text = row["experience"]
            matches = list(re.finditer(date_regex, text, re.IGNORECASE))
            if not matches:
                return f"[[role: {row['title']}][0 years][content: {text}]]"

            blocks = []
            for m in matches:
                dur = self.calculate_duration(m.group(0))
                desc = re.sub(date_regex, "", text, flags=re.IGNORECASE)
                blocks.append(f"[[role: {row['title']}][{dur} years][content: {desc.strip()}]]")
            return " ".join(blocks)

        df["experience_enriched"] = df.apply(process, axis=1)
        return df

    # =========================
    # EDUCATION ENRICH
    # =========================
    def enrich_education(self, text):
        if not isinstance(text, str):
            return "[[institution: unknown][cert_count: 0][content: ]]"

        certs = len(re.findall(r"\b(certified|certificate|certification|license|cpa|cfa)\b", text, re.I))
        inst = re.findall(r"((?:\b\w+\b\s+){1,3}(university|college|institute|school|polytechnic|universitas))", text, re.I)
        inst = ", ".join({i[0].strip() for i in inst}) or "unknown"

        clean = re.sub(r"\b\d{4}\b|gpa.*", "", text, flags=re.I)
        clean = re.sub(self.MONTHS_PATTERN, "", clean, flags=re.I)
        clean = re.sub(r"\s+", " ", clean).strip()

        return f"[[institution: {inst}][cert_count: {certs}][content: {clean}]]"

    # =========================
    # PIPELINE
    # =========================
    def run(self, pdf_folder: str) -> pd.DataFrame:
        rows = []

        for pdf in Path(pdf_folder).glob("*.pdf"):
            text = self.pdf_to_text(pdf)
            feat = self.extract_features(text)
            feat["cv_id"] = pdf.name

            for k in feat:
                feat[k] = self.clean_line(feat[k])

            if not feat["title"]:
                feat["title"] = self.infer_title_from_experience(feat["experience"])

            feat["skills_list"] = [s.strip() for s in feat["skills"].split(",") if s.strip()]
            rows.append(feat)

        df = pd.DataFrame(rows)

        df = self.enrich_experience(df)
        df["education_enriched"] = df["education"].apply(self.enrich_education)

        return df


In [ ]:
path = "/content/drive/MyDrive/resume-dataset/data/data/ACCOUNTANT"

pipeline = CVPipeline()

try:
    df_f = pipeline.run(path)
    display(df_f.head())
except Exception as e:
    print("Pipeline error:", e)

,title,summary,experience,skills,education,cv_id,skills_list,experience_enriched,education_enriched
0,financial accountant,client-focused sales representative with 8+ ye...,"financial accountant, 11/2017 to 03/2018 compa...","usgaap principles, cash flow analysis, account...","bachelor of science : accounting, 1993 rhode i...",29821051.pdf,"[usgaap principles, cash flow analysis, accoun...",[[role: financial accountant][0.3 years][conte...,[[institution: 1993 rhode island college][cert...
1,accountant,accomplished professional with exceptional ski...,01/2006 to 01/2014 accountant company name cit...,"financial statements, general ledgers analysis...",02/2004 master of business administration : ac...,23139819.pdf,"[financial statements, general ledgers analysi...",[[role: accountant][8.0 years][content: accoun...,[[institution: business administration parklan...
2,accountant,"a detail oriented, efficient accountant that e...","company name city, state accountant 12/2012 to...","account reconciliation expert, financial model...",bachelor of science : accounting and informati...,42487883.pdf,"[account reconciliation expert, financial mode...",[[role: accountant][13.2 years][content: compa...,[[institution: and information management univ...
3,accountant,"experienced, detail-oriented accountant who ef...",accountant sep 2012 to current company name ci...,"billing and collections ms office suite, accou...","associate of applied science, accounting st. l...",25127518.pdf,"[billing and collections ms office suite, acco...",[[role: accountant][13.4 years][content: accou...,[[institution: louis community college][cert_c...
4,general accountant,"team-oriented accountant, successful at managi...",01/2016 to 11/2016 general accountant treasury...,"accounting, ap, ar, balance sheet, benefits, b...",bachelor of science : accounting upper iowa un...,36024962.pdf,"[accounting, ap, ar, balance sheet, benefits, ...",[[role: general accountant][0.8 years][content...,[[institution: accounting upper iowa universit...


In [ ]:
import re
import ast
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util


class CVScorer:
    def __init__(
        self,
        job_title: str,
        job_description: str,
        required_skills: list,
        highlight_keywords: list,
        weights: dict,
        model_name: str = "all-MiniLM-L6-v2",
        title_sim_threshold: float = 0.6,
    ):
        self.job_title = job_title
        self.job_description = job_description
        self.required_skills = required_skills
        self.highlight_keywords = highlight_keywords
        self.weights = weights
        self.title_sim_threshold = title_sim_threshold

        self.model = SentenceTransformer(model_name)

        # Pre-encode target (optimasi)
        self.job_title_emb = self.model.encode(job_title, convert_to_tensor=True)
        self.job_desc_emb = self.model.encode(job_description, convert_to_tensor=True)

    # ======================================================
    # GATE: TITLE FILTER
    # ======================================================
    def filter_by_title(self, df: pd.DataFrame) -> pd.DataFrame:
        def _pass(title):
            if not title or pd.isna(title):
                return False
            t_cv = str(title).lower().strip()
            t_job = self.job_title.lower().strip()

            if t_job in t_cv or t_cv in t_job:
                return True

            emb_cv = self.model.encode(t_cv, convert_to_tensor=True)
            sim = util.pytorch_cos_sim(emb_cv, self.job_title_emb).item()
            return sim >= self.title_sim_threshold

        return df[df["title"].apply(_pass)].copy().reset_index(drop=True)

    # ======================================================
    # SKILLS
    # ======================================================
    def score_skills(self, cv_skills_raw) -> float:
        if not self.required_skills:
            return 0.0
        try:
            cv_skills = ast.literal_eval(cv_skills_raw) if isinstance(cv_skills_raw, str) else cv_skills_raw
        except:
            cv_skills = []

        if not cv_skills:
            return 0.0

        cv_low = [str(s).lower().strip() for s in cv_skills]
        req_low = [s.lower().strip() for s in self.required_skills]
        n = len(req_low)

        hard = []
        remain = []

        for s in req_low:
            (hard if s in cv_low else remain).append(s)

        score = len(hard)

        if remain:
            emb_cv = self.model.encode(cv_low, convert_to_tensor=True)
            cv_used = set()

            for s in remain:
                emb_s = self.model.encode(s, convert_to_tensor=True)
                sims = util.pytorch_cos_sim(emb_s, emb_cv)[0]

                for idx in cv_used:
                    sims[idx] = -1

                best_idx = torch.argmax(sims).item()
                best_sim = sims[best_idx].item()

                if best_sim > 0:
                    score += best_sim
                    cv_used.add(best_idx)

        return round(score / n, 4)

    # ======================================================
    # SUMMARY
    # ======================================================
    def score_summary_raw(self, summary) -> float:
        if not summary or pd.isna(summary):
            return 0.0

        def clean(text):
            fluff = [
                "professional", "dedicated", "hardworking", "seeking",
                "opportunity", "proven", "years", "experience"
            ]
            text = str(text).lower()
            for w in fluff:
                text = re.sub(rf"\b{w}\b", "", text)
            return re.sub(r"\s+", " ", text).strip()

        chunks = [c.strip() for c in str(summary).replace("\n", ".").split(".") if len(c.strip()) > 10]
        if not chunks:
            return 0.0

        emb_chunks = self.model.encode([clean(c) for c in chunks], convert_to_tensor=True)
        sims = util.pytorch_cos_sim(emb_chunks, self.job_desc_emb).flatten().tolist()
        score = max(sims) if sims else 0.0

        bonus = sum(0.5 for kw in self.highlight_keywords if kw.lower() in summary.lower())
        return score + bonus

    # ======================================================
    # EDUCATION
    # ======================================================
    def score_education_raw(self, edu) -> float:
        if not edu or pd.isna(edu):
            return 0.0

        try:
            cert = int(re.search(r"cert_count:\s*(\d+)", edu).group(1))
            content = re.search(r"content:\s*(.*?)\]\]", edu).group(1)
        except:
            return 0.0

        degree_weights = {
            "phd": 2.0, "doctorate": 2.0,
            "master": 1.5, "mba": 1.5,
            "bachelor": 1.2, "ba": 1.2, "bs": 1.2,
            "diploma": 1.0, "d3": 1.0,
            "high school": 0.5,
        }

        weight = 0.5
        for d, w in degree_weights.items():
            if d in content.lower():
                weight = max(weight, w)

        emb = self.model.encode(content, convert_to_tensor=True)
        sim = max(0, util.pytorch_cos_sim(emb, self.job_desc_emb).item())

        return (sim * weight) + (cert * 0.1)

    # ======================================================
    # EXPERIENCE
    # ======================================================
    def score_experience_raw(self, exp) -> float:
        if not exp or pd.isna(exp):
            return 0.0

        blocks = re.findall(r"\[\[(.*?)\]\]", exp, re.DOTALL)
        if not blocks:
            return 0.0

        total = 0.0

        for blk in blocks:
            parts = blk.split("][")
            if len(parts) < 3:
                continue

            role = parts[0].replace("role:", "").strip()
            years = float(re.findall(r"[\d.]+", parts[1])[0]) if re.findall(r"[\d.]+", parts[1]) else 1.0
            content = parts[2].replace("content:", "").strip()

            duration = np.log1p(years) + 1
            role_sim = util.pytorch_cos_sim(
                self.model.encode(role, convert_to_tensor=True),
                self.job_title_emb
            ).item()

            chunks = [c for c in content.split(".") if len(c.strip()) > 15]
            content_score = 0.0
            if chunks:
                embs = self.model.encode(chunks, convert_to_tensor=True)
                sims = sorted(util.pytorch_cos_sim(embs, self.job_desc_emb).flatten().tolist(), reverse=True)
                content_score = max(0, sims[0]) + sum(s * 0.2 for s in sims[1:] if s > 0.5)

            kw_bonus = sum(0.2 for kw in self.highlight_keywords if kw.lower() in content.lower())
            relevance = (max(0, role_sim) * 5) + (content_score * 3) + kw_bonus

            total += relevance * duration

        return round(total, 4)

    # ======================================================
    # PIPELINE UTAMA
    # ======================================================
    def score_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        df = self.filter_by_title(df)
        if df.empty:
            return df

        df["score_skills"] = df["skills_list"].apply(self.score_skills)
        df["summary_raw"] = df["summary"].apply(self.score_summary_raw)
        df["edu_raw"] = df["education_enriched"].apply(self.score_education_raw)
        df["exp_raw"] = df["experience_enriched"].apply(self.score_experience_raw)

        norm_map = {
            "summary_raw": "score_summary_final",
            "edu_raw": "score_education_final",
            "exp_raw": "score_experience_final",
        }

        for r, f in norm_map.items():
            mn, mx = df[r].min(), df[r].max()
            df[f] = (df[r] - mn) / (mx - mn) if mx != mn else 0.5

        df["total_score"] = (
            df["score_experience_final"] * self.weights["experience"] +
            df["score_skills"] * self.weights["skills"] +
            df["score_summary_final"] * self.weights["summary"] +
            df["score_education_final"] * self.weights["education"]
        )

        return df.sort_values("total_score", ascending=False).reset_index(drop=True)


In [ ]:
# ==========================================
# CONTOH INISIALISASI SCORER
# ==========================================

job_title = "Senior Accountant / Tax Supervisor"

job_desc_input = """
Handle full set of accounts, monthly closing, and financial reporting.
Experienced in corporate tax compliance (VAT, PPh), auditing, and financial analysis.
Familiar with IFRS and SAP system.
"""

input_user_skills = [
    "accounting",
    "microsoft excel",
    "financial reporting",
    "taxation"
]

highlights = [
    "SAP",
    "IFRS",
    "Tax Compliance",
    "Auditing",
    "CPA",
    "Tax Planning"
]

scorer = CVScorer(
    job_title=job_title,
    job_description=job_desc_input,
    required_skills=input_user_skills,
    highlight_keywords=highlights,
    weights={
        "experience": 0.5,
        "skills": 0.3,
        "summary": 0.1,
        "education": 0.1
    }
)
df_ranked = scorer.score_dataframe(df_f)

df_ranked[["cv_id", "title", "total_score"]].head(10)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,cv_id,title,total_score
0,17306905.pdf,senior accountant,0.835231
1,29050809.pdf,accountant ii,0.655275
2,27637576.pdf,corporate accountant,0.651678
3,21763056.pdf,accountant,0.627849
4,23513618.pdf,accountant,0.593250
5,24103168.pdf,financial accountant,0.554876
6,15906625.pdf,accountant,0.554451
7,25846894.pdf,accountant,0.550321
8,63137898.pdf,accountant,0.550187
9,33527446.pdf,accountant,0.547189


In [ ]:
df_ranked

,title,summary,experience,skills,education,cv_id,skills_list,experience_enriched,education_enriched,score_skills,summary_raw,edu_raw,exp_raw,score_summary_final,score_education_final,score_experience_final,total_score
0,senior accountant,financial and accounting professional with exp...,"senior accountant, 01/2019 to current company ...",power user of microsoft excel epicor netsuite ...,bachelor of business administration : accounti...,17306905.pdf,[power user of microsoft excel epicor netsuite...,[[role: senior accountant][7.1 years][content:...,[[institution: 2014 valdosta state university]...,0.8949,0.505426,0.547564,208.2042,0.332717,0.334889,1.000000,0.835231
1,accountant ii,a professional accountant with more than 10 ye...,accountant ii 10/2016 to current company name ...,month-end close activities - reconciliations/a...,certified public accountant (cpa licensed in t...,29050809.pdf,[month-end close activities - reconciliations/...,[[role: accountant ii][9.3 years][content: acc...,[[institution: 2001 university][cert_count: 2]...,0.9148,0.541461,0.788668,124.4661,0.356438,0.515407,0.587301,0.655275
2,corporate accountant,strategic and analytical finance professional ...,corporate accountant may 2015 to march 2016 co...,"superior time management, financial modeling, ...",high school diploma : business management/acco...,27637576.pdf,"[superior time management, financial modeling,...",[[role: corporate accountant][0.8 years][conte...,[[institution: accounting zephyrhills high sch...,0.4059,0.525886,0.357058,198.4906,0.346186,0.192255,0.952127,0.651678
3,accountant,to obtain a challenging position in a professi...,accountant march 2001 to current company name ...,"account reconciliation, accounting, accounts p...",master degree : accounting gujarat university ...,21763056.pdf,"[account reconciliation, accounting, accounts ...",[[role: accountant][24.9 years][content: accou...,"[[institution: accounting gujarat university, ...",0.9449,1.062428,0.645331,100.1101,0.699386,0.408088,0.467263,0.627849
4,accountant,results-oriented and organized bilingual accou...,"01/2014 to current company name city, state a ...","accounting, accountant, accounts receivable, a...",master of business administration : accounting...,23513618.pdf,"[accounting, accountant, accounts receivable, ...",[[role: accountant][12.1 years][content: compa...,"[[institution: robert morris university, 1 201...",0.8864,1.519087,0.874737,74.0224,1.000000,0.579848,0.338691,0.593250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,accountant,reliable customer service representative with ...,01/2014 to 06/2015 accountant company name - c...,"strong organizational skills, active listening...",2013 associate of arts : international busines...,25935030.pdf,"[strong organizational skills, active listenin...",[[role: accountant][1.4 years][content: accoun...,"[[institution: business school, international ...",0.4899,0.396014,0.104226,33.6526,0.260692,0.002956,0.139731,0.243200
101,accountant,financial accountant specializing in financial...,accountant july 2012 to october 2015 company n...,"periodic financial reporting expert, invoice c...",select one bachelor of arts : business studies...,16237710.pdf,"[periodic financial reporting expert, invoice ...",[[role: accountant][3.2 years][content: accoun...,"[[institution: 2014 university, 2013 oshwal co...",0.5175,0.580331,0.529277,12.0755,0.382026,0.321197,0.033389,0.242267
102,accountant,self-motivated accountant offering a strong wo...,03/2010 to current accountant company name cit...,"accounts receivable professional, sales softwa...",2009 bachelor of science : accounting kaplan u...,18365791.pdf,"[accounts receivable professional, sales softw...",[[role: accountant][15.9 years][content: accou...,[[institution: accounting kaplan university][c...,0.3123,0.383664,0.511796,39.8822,0.252562,0.308109,0.170433,0.234974
103,"supervisor, accountant",motivated sales professional with 10+ years sa...,"01/2

In [ ]:
import torch
import transformers
import sentence_transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)


torch: 2.10.0+cu128
transformers: 5.0.0
sentence-transformers: 5.2.3


# CV Text Cleaning & Feature Extraction Pipeline

This document describes the CV preprocessing pipeline in a structured, rule-based format.

---

## 1. Input

- Input format: PDF files (`*.pdf`)
- Each PDF represents one CV
- Output format: tabular data (DataFrame)

---

## 2. PDF to Raw Text

**Operation**
- Read each PDF page
- Extract plain text
- Concatenate all pages

**Result**
- One raw text string per CV

---

## 3. Line Cleaning Rules

Each line of text is normalized using the following rules:

1. Remove bullet symbols  
   (`• ● ○ ▪ *` → `,`)

2. Remove non-ASCII characters

3. Convert text to lowercase

4. Replace `&` with `and`

5. Normalize whitespace  
   (multiple spaces → single space)

6. Normalize commas  
   (`, , ,` → `,`)

7. Trim leading and trailing punctuation

**Output**
- Clean, lowercase, ASCII-only lines

---

## 4. Section Header Detection

A line is considered a **section header** if:

- Length ≤ 40 characters
- Does NOT end with a period
- Contains ≤ 5 words

---

## 5. Header-to-Feature Mapping

Detected headers are mapped to CV features:

| Feature     | Possible Headers |
|------------|------------------|
| Title      | title, role, position |
| Summary    | summary, professional summary, profile |
| Experience | experience, work history, employment history |
| Skills     | skills, technical skills, highlights |
| Education  | education, academic background |

---

## 6. Feature Extraction Logic

Processing is done line-by-line:

1. Start with current section = `title`
2. If a line matches a known header:
   - Switch active section
3. Otherwise:
   - Append line content to the active section

Special rules:
- First non-header line is assumed to be the **job title**
- Skills are concatenated using commas
- Other sections are concatenated as sentences

---

## 7. Title Fallback Rule

If no explicit title is found:

- Extract the first phrase from experience section
- Remove year numbers
- Use first 3–4 words as inferred title

---

## 8. Skill Parsing

- Skills section is split by commas
- Each skill is trimmed
- Empty values are removed

**Output**
- `skills_list`: list of skill strings

---

## 9. Experience Enrichment

### 9.1 Date Detection

Recognized date formats:
- `Month YYYY`
- `MM/YYYY`
- `YYYY`
- `current`, `present`, `now`

### 9.2 Duration Calculation

Steps:
1. Convert dates to decimal years
2. Compute absolute difference
3. If only one date found → duration = 0.5 years

### 9.3 Experience Block Format

Each experience is converted into:

```

[[role: <title>][<duration> years][content: <description>]]

```

If no dates are detected:
- Duration defaults to 0 years

---

## 10. Education Enrichment

From education text, extract:

### 10.1 Certification Count
Keywords:
- certified
- certificate
- certification
- license
- cpa
- cfa

### 10.2 Institution Name
Detected if containing:
- university
- college
- institute
- school
- polytechnic

### 10.3 Content Cleaning
- Remove years
- Remove GPA information
- Remove month names

### 10.4 Education Block Format

```

[[institution: <name>][cert_count: <number>][content: <cleaned text>]]

```

---

## 11. Final Output Schema

Each CV produces one row with:

- cv_id
- title
- summary
- experience
- experience_enriched
- skills
- skills_list
- education
- education_enriched

---

## 12. Pipeline Summary

```

PDF
↓
Raw Text
↓
Line Cleaning
↓
Header Detection
↓
Feature Extraction
↓
Experience Enrichment
↓
Education Enrichment
↓
Structured CV Data

```

---

This pipeline is fully rule-based, deterministic, and model-free.

# CV Scoring Mathematical Formulation (Kaggle Compatible)

## General Notation

- CV index: i  
- T_i : CV title  
- J_T : job title  
- J_D : job description  
- S_i : CV skill list  
- S_r : required skill list  
- K : highlight keywords  
- E_i : education text  
- X_i : experience text  
- φ(x) : sentence embedding  
- cos(u, v) : cosine similarity  
- τ : title similarity threshold  

---

## 1. Title Gate Filter

$$
Pass(T_i) =
\begin{cases}
1, & \text{if } J_T \text{ in } T_i \text{ or } T_i \text{ in } J_T \\
1, & \text{if } cos(\phi(T_i), \phi(J_T)) \ge \tau \\
0, & \text{otherwise}
\end{cases}
$$

Only CVs with `Pass(T_i) = 1` are evaluated.

---

## 2. Skill Score

Exact matched skills:

$$
H_i = \{ s \mid s \in S_r \text{ and } s \in S_i \}
$$

Unmatched required skills:

$$
R_i = S_r \setminus H_i
$$

Hard score:

$$
S_{hard} = |H_i|
$$

Semantic score:

$$
S_{sem} =
\sum_{s \in R_i}
\max_{c \in S_i}
cos(\phi(s), \phi(c)), \quad cos > 0
$$

Final skill score:

$$
Score_{skills}(i) =
\frac{S_{hard} + S_{sem}}{|S_r|}
$$

---

## 3. Summary Score

Summary sentences:

$$
C_i = \{c_1, c_2, \dots, c_n\}
$$

Similarity score:

$$
S_{sim} = \max_{c \in C_i} cos(\phi(c), \phi(J_D))
$$

Keyword bonus:

$$
B_{sum} = 0.5 \sum_{k \in K} I(k \text{ in summary}_i)
$$

Final summary score:

$$
Score_{summary}(i) = S_{sim} + B_{sum}
$$

---

## 4. Education Score

Degree weight:

$$
w_d = \max(w_j)
$$

Education similarity:

$$
S_{edu} = \max(0, cos(\phi(E_i), \phi(J_D)))
$$

Certification count:

$$
C_i = cert\_count
$$

Final education score:

$$
Score_{edu}(i) = w_d \cdot S_{edu} + 0.1 \cdot C_i
$$

---

## 5. Experience Score

For each experience block k:

Role similarity:

$$
R_k = \max(0, cos(\phi(role_k), \phi(J_T)))
$$

Duration weight:

$$
D_k = \log(1 + y_k) + 1
$$

Content relevance:

$$
C_k = \max(s_1) + \sum_{j>1} 0.2 \cdot s_j \quad \text{where } s_j > 0.5
$$

Keyword bonus:

$$
B_k = 0.2 \sum_{h \in K} I(h \text{ in content}_k)
$$

Block relevance:

$$
Q_k = 5R_k + 3C_k + B_k
$$

Total experience score:

$$
Score_{exp}(i) = \sum_k Q_k \cdot D_k
$$

---

## 6. Min–Max Normalization

For feature f:

$$
\hat{f}_i =
\begin{cases}
\frac{f_i - \min(f)}{\max(f) - \min(f)}, & \max(f) \ne \min(f) \\
0.5, & \text{otherwise}
\end{cases}
$$

---

## 7. Total Score

Weights:

$$
(w_{exp}, w_{skills}, w_{summary}, w_{edu})
$$

Final score:

$$
TotalScore_i =
w_{exp}\hat{E}_i +
w_{skills}Score_{skills}(i) +
w_{summary}\hat{S}_i +
w_{edu}\hat{D}_i
$$

---

## 8. Ranking Rule

CVs are ranked in descending order of `TotalScore`.


In [ ]:
job_vacancy = {
    "job_id": "JOB-ACC-001",
    "job_title": "Senior Accountant / Tax Supervisor",
    "job_description": """
    Handle full set of accounts, monthly closing, and financial reporting.
    Experienced in corporate tax compliance (VAT, PPh), auditing, and financial analysis.
    Familiar with IFRS and SAP system.
    """,
    "required_skills": ["accounting", "microsoft excel", "financial reporting", "taxation"],
    "highlight_keywords": ["SAP", "IFRS", "Tax Compliance", "Auditing", "CPA", "Tax Planning"],
    "weights": {"experience": 0.5, "skills": 0.3, "summary": 0.1, "education": 0.1},
    "shortlist_threshold": 0.60
}

In [ ]:
from google.colab import files
import os, shutil, glob

uploaded = files.upload()

tmp_folder = "/content/incoming_cv"
os.makedirs(tmp_folder, exist_ok=True)

# clear folder
for f in glob.glob(tmp_folder + "/*"):
    os.remove(f)

# move uploaded pdf into folder
pdf_name = list(uploaded.keys())[0]
shutil.move(pdf_name, f"{tmp_folder}/{pdf_name}")

Saving 13294301.pdf to 13294301.pdf


'/content/incoming_cv/13294301.pdf'

In [ ]:
import glob, os

pdfs = glob.glob(tmp_folder + "/*.pdf")
print("PDFs found:", len(pdfs))
print([os.path.basename(p) for p in pdfs])

PDFs found: 1
['13294301.pdf']


In [ ]:
# Parse
pipeline = CVPipeline()
df_cv = pipeline.run(tmp_folder)

print("Parsed shape:", df_cv.shape)
print("Parsed columns:", df_cv.columns.tolist())
display(df_cv.head())

Parsed shape: (1, 9)
Parsed columns: ['title', 'summary', 'experience', 'skills', 'education', 'cv_id', 'skills_list', 'experience_enriched', 'education_enriched']


,title,summary,experience,skills,education,cv_id,skills_list,experience_enriched,education_enriched
0,accountant,emerging accounting professional ready to deve...,"accountant, 04/2020 to current company name ci...","microsoft office, account reconciliation proce...","bachelor of science : accounting and finance, ...",13294301.pdf,"[microsoft office, account reconciliation proc...",[[role: accountant][5.8 years][content: accoun...,"[[institution: 2020 oakland university, 2016 m..."


In [ ]:
scorer = CVScorer(
    job_title=job_vacancy["job_title"],
    job_description=job_vacancy["job_description"],
    required_skills=job_vacancy["required_skills"],
    highlight_keywords=job_vacancy["highlight_keywords"],
    weights=job_vacancy["weights"]
)

df_scored = scorer.score_dataframe(df_cv)

df_scored["shortlisted"] = df_scored["total_score"] >= job_vacancy["shortlist_threshold"]

df_scored[["cv_id", "title", "total_score", "shortlisted"]]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,cv_id,title,total_score,shortlisted
0,13294301.pdf,accountant,0.50417,False


In [ ]:
# ==========================================================
# HireNext.ai — Professional CV Screening (Semantic Scoring)
# Upload (up to 10 PDFs) → Parse → Score → Ranked Report + Shortlist
# - Safe against KeyError
# - Works with multiple PDFs
# - Shows detailed score breakdown
# ==========================================================

from google.colab import files
import os, shutil, glob
import pandas as pd

# -------------------------
# 0) Job config (HR Generalist)
# -------------------------
JOB_VACANCY = {
    "job_id": "JOB-HR-001",
    "job_title": "HR Generalist",
    "job_description": """
Support end-to-end HR operations including recruitment, onboarding, employee relations,
HR documentation, HRIS updates, and compliance. Coordinate training, handle HR queries,
and assist performance management and payroll coordination.
""".strip(),
    "required_skills": [
        "recruitment", "onboarding", "employee relations",
        "hr administration", "hris", "labor law", "microsoft excel"
    ],
    "highlight_keywords": [
        "HRIS", "PeopleSoft", "Workday", "SAP SuccessFactors", "Compliance",
        "Performance Management", "Training", "Payroll", "Onboarding", "Policy"
    ],
    "weights": {"experience": 0.5, "skills": 0.3, "summary": 0.1, "education": 0.1},
    "shortlist_threshold": 0.60
}

# -------------------------
# 1) Settings
# -------------------------
TMP_FOLDER = "/content/incoming_cv"
MAX_UPLOADS = 10

# Title gate helps reject irrelevant CVs, but HR titles vary a lot.
USE_TITLE_GATE = True
TITLE_SIM_THRESHOLD = 0.45   # try 0.40–0.55 for HR roles

# -------------------------
# 2) Helpers
# -------------------------
def prepare_upload_folder(folder: str) -> None:
    os.makedirs(folder, exist_ok=True)
    for f in glob.glob(os.path.join(folder, "*")):
        try:
            os.remove(f)
        except:
            pass

def upload_pdfs_to_folder(folder: str, max_uploads: int = 10) -> list[str]:
    uploaded = files.upload()  # select multiple PDFs
    pdf_names = [n for n in uploaded.keys() if n.lower().endswith(".pdf")]

    if not pdf_names:
        raise ValueError("❌ No PDFs uploaded. Please upload one or more PDF files.")

    if len(pdf_names) > max_uploads:
        print(f"⚠️ You selected {len(pdf_names)} PDFs. Using first {max_uploads} only.")
        pdf_names = pdf_names[:max_uploads]

    for name in pdf_names:
        shutil.move(name, os.path.join(folder, name))

    return pdf_names

def bypass_title_gate_score_all(df_cv: pd.DataFrame, scorer, weights: dict) -> pd.DataFrame:
    """Score all CVs without filtering by title gate (uses same scoring logic)."""
    df = df_cv.copy()

    df["score_skills"] = df["skills_list"].apply(scorer.score_skills)
    df["summary_raw"] = df["summary"].apply(scorer.score_summary_raw)
    df["edu_raw"] = df["education_enriched"].apply(scorer.score_education_raw)
    df["exp_raw"] = df["experience_enriched"].apply(scorer.score_experience_raw)

    # Min-max normalize (batch-based). If only 1 CV => becomes 0.5 by design.
    norm_map = {
        "summary_raw": "score_summary_final",
        "edu_raw": "score_education_final",
        "exp_raw": "score_experience_final",
    }

    for raw_col, final_col in norm_map.items():
        mn, mx = df[raw_col].min(), df[raw_col].max()
        df[final_col] = (df[raw_col] - mn) / (mx - mn) if mx != mn else 0.5

    df["total_score"] = (
        df["score_experience_final"] * weights["experience"] +
        df["score_skills"] * weights["skills"] +
        df["score_summary_final"] * weights["summary"] +
        df["score_education_final"] * weights["education"]
    )

    return df.sort_values("total_score", ascending=False).reset_index(drop=True)

def make_report(df_scored: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df_scored.copy()
    df["shortlisted"] = df["total_score"] >= threshold

    report_cols = [
        "cv_id", "title",
        "score_skills",
        "score_experience_final", "score_summary_final", "score_education_final",
        "total_score", "shortlisted",
        "skills_list",
        "exp_raw", "summary_raw", "edu_raw"
    ]
    report_cols = [c for c in report_cols if c in df.columns]

    df_report = df[report_cols].sort_values("total_score", ascending=False).reset_index(drop=True)
    df_shortlisted = df_report[df_report["shortlisted"] == True].reset_index(drop=True)
    return df_report, df_shortlisted

# -------------------------
# 3) Upload PDFs
# -------------------------
prepare_upload_folder(TMP_FOLDER)
picked = upload_pdfs_to_folder(TMP_FOLDER, MAX_UPLOADS)

pdfs = glob.glob(os.path.join(TMP_FOLDER, "*.pdf"))
print("PDFs in folder:", len(pdfs))
print([os.path.basename(p) for p in pdfs])

# -------------------------
# 4) Parse PDFs → DataFrame (requires your CVPipeline class already defined)
# -------------------------
pipeline = CVPipeline()
df_cv = pipeline.run(TMP_FOLDER)

print("Parsed shape:", df_cv.shape)  # should be (N, 9) when N PDFs uploaded
print("Parsed columns:", df_cv.columns.tolist())
display(df_cv[["cv_id", "title"]].head(10))

# -------------------------
# 5) Score + Shortlist (requires your CVScorer class already defined)
# -------------------------
scorer = CVScorer(
    job_title=JOB_VACANCY["job_title"],
    job_description=JOB_VACANCY["job_description"],
    required_skills=JOB_VACANCY["required_skills"],
    highlight_keywords=JOB_VACANCY["highlight_keywords"],
    weights=JOB_VACANCY["weights"],
    title_sim_threshold=TITLE_SIM_THRESHOLD
)

if USE_TITLE_GATE:
    df_scored = scorer.score_dataframe(df_cv)
    # If title gate filters everything, fallback to scoring all (professional behavior)
    if df_scored.empty:
        print("⚠️ Title gate filtered all CVs. Falling back to scoring ALL CVs (no title gate).")
        df_scored = bypass_title_gate_score_all(df_cv, scorer, JOB_VACANCY["weights"])
else:
    df_scored = bypass_title_gate_score_all(df_cv, scorer, JOB_VACANCY["weights"])

# Safety check
if df_scored is None or df_scored.empty or "total_score" not in df_scored.columns:
    raise RuntimeError("❌ Scoring failed. Please check CVPipeline/CVScorer definitions.")

# -------------------------
# 6) Final Ranked Report + Shortlisted Only
# -------------------------
df_report, df_shortlisted = make_report(df_scored, JOB_VACANCY["shortlist_threshold"])

print("\n📌 Ranked CV Report (Top results first):")
display(df_report)

print("\n✅ Shortlisted only:")
display(df_shortlisted)

Saving 51769822.pdf to 51769822.pdf
Saving 52979663.pdf to 52979663.pdf
Saving 57667857.pdf to 57667857.pdf
Saving 59962788.pdf to 59962788.pdf
Saving 72231872.pdf to 72231872.pdf
Saving 73077810.pdf to 73077810.pdf
Saving 80162314.pdf to 80162314.pdf
Saving 86184722.pdf to 86184722.pdf
Saving 87968870.pdf to 87968870.pdf
Saving 91930382.pdf to 91930382.pdf
Saving 93002334.pdf to 93002334.pdf
Saving 93112113.pdf to 93112113.pdf
⚠️ You selected 12 PDFs. Using first 10 only.
PDFs in folder: 10
['51769822.pdf', '91930382.pdf', '59962788.pdf', '86184722.pdf', '72231872.pdf', '87968870.pdf', '80162314.pdf', '57667857.pdf', '73077810.pdf', '52979663.pdf']
Parsed shape: (10, 9)
Parsed columns: ['title', 'summary', 'experience', 'skills', 'education', 'cv_id', 'skills_list', 'experience_enriched', 'education_enriched']


,cv_id,title
0,51769822.pdf,hr specialist
1,91930382.pdf,hr intern
2,59962788.pdf,hr executive
3,86184722.pdf,hr generalist
4,72231872.pdf,hr generalist
5,87968870.pdf,hr generalist
6,80162314.pdf,hr professional
7,57667857.pdf,hr consultant
8,73077810.pdf,hr generalist/recruiter
9,52979663.pdf,senior hr


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📌 Ranked CV Report (Top results first):


,cv_id,title,score_skills,score_experience_final,score_summary_final,score_education_final,total_score,shortlisted,skills_list,exp_raw,summary_raw,edu_raw
0,72231872.pdf,hr generalist,0.8260,1.000000,1.000000,0.550787,0.902879,True,"[accounts payable, accounts receivable, ada, a...",159.0194,2.096589,0.596678
1,57667857.pdf,hr consultant,0.6803,0.831331,0.187340,0.350302,0.673520,True,[phr certified professionals in human resource...,134.4401,0.392774,0.417435
2,87968870.pdf,hr generalist,0.7338,0.648481,0.297037,0.479898,0.622074,True,"[hiring and retention, training and developmen...",107.7943,0.622765,0.533300
3,52979663.pdf,senior hr,0.6396,0.689814,0.000000,0.154282,0.552215,False,[safety managementemployee engagementhr genera...,113.8175,0.000000,0.242183
4,86184722.pdf,hr generalist,0.6584,0.595519,0.000000,0.434539,0.538733,False,"[microsoft office suite, sap, kronos, ibm, adp...",100.0764,0.000000,0.492747
5,73077810.pdf,hr generalist/recruiter,0.7349,0.362865,0.177282,0.033889,0.423020,False,"[adp, people fluent, microsoft offices, interv...",66.1729,0.371688,0.134546
6,51769822.pdf,hr specialist,0.6890,0.350721,0.210516,0.000000,0.403112,False,"[administration/, accounting/hr., administrati...",64.4032,0.441365,0.104248
7,80162314.pdf,hr professional,0.5902,0.311177,0.145569,0.500432,0.397248,False,"[staff recruitment and retention, employee rel...",58.6406,0.305199,0.551659
8,59962788.pdf,hr executive,0.5413,0.000000,0.170856,1.000000,0.279476,False,"[new employee orientations, compensation and b...",13.2944,0.358216,0.998297
9,91930382.pdf,hr intern,0.2551,0.249708,0.153057,0.625129,0.279203,False,"[bullhorn, boolean searches, google resume sea...",49.6831,0.320897,0.663144



✅ Shortlisted only:


,cv_id,title,score_skills,score_experience_final,score_summary_final,score_education_final,total_score,shortlisted,skills_list,exp_raw,summary_raw,edu_raw
0,72231872.pdf,hr generalist,0.8260,1.000000,1.000000,0.550787,0.902879,True,"[accounts payable, accounts receivable, ada, a...",159.0194,2.096589,0.596678
1,57667857.pdf,hr consultant,0.6803,0.831331,0.187340,0.350302,0.673520,True,[phr certified professionals in human resource...,134.4401,0.392774,0.417435
2,87968870.pdf,hr generalist,0.7338,0.648481,0.297037,0.479898,0.622074,True,"[hiring and retention, training and developmen...",107.7943,0.622765,0.533300
